# Phase 4 (Court Judgment Prediction, CJPE) — Run #1 on Phase-1 **Decoder-only model 02** (hybrid local/global attention + GQA), with **LoRMA**

Config-driven like the Phase-2 / Phase-3 notebooks: edit **Cell 2** only. This is the **first** Phase-4 run of the plan (Dec-02 + **LoRMA**, the compulsory technique) and it is also the planned **KD teacher** (`EXPORT_SOFT_LABELS=True`). The same notebook produces the controls you need for the report (`DOC_VIEW`, `AGG`, `METHOD`, `ATTN_MODE`).

### The task in one paragraph
Each case is one long judgment (`{"id", "text"}`; train/dev also have a binary label). Predict **0 or 1** per case. Metric: **macro-F1** (so the minority outcome weighs as much as the majority one). The test file must give **exactly one integer prediction per test id**.

### Why a hierarchical (chunked) model
The Phase-1 context is 1024 tokens; judgments are far longer. Full attention over a document of `N = K·L` tokens costs `O((KL)²)`; encoding `K` chunks of `L` tokens independently costs `O(K·L²)`, i.e. `K` times less, and chunks can be batched. So:

1. **Select** at most `MAX_CHUNKS` chunks of `CHUNK_TOKENS` tokens from the document (`DOC_VIEW`, see below).
2. **Encode** every chunk with the Phase-1 backbone (adapted by LoRMA) and **mean-pool** its tokens: `h_c = mean_t H_{c,t}` (`H` = final hidden states, `d = 512`).
3. **Aggregate** the chunk vectors into one document vector (`AGG`).
4. `dropout → Linear(512 → 2)`; class-weighted cross-entropy; the **decision threshold is tuned on dev** (macro-F1 needs a good operating point, not only a good ranking).

| `DOC_VIEW` | chunks used | question it answers |
|---|---|---|
| `"tail"` (default) | the **last** `MAX_CHUNKS·(CHUNK_TOKENS−1)` tokens (chunks aligned to the end of the case) | judgments usually end with the reasoning and the order, so the end may carry most of the signal |
| `"head"` | the first ones | facts / procedural history only |
| `"head_tail"` | half from the start, half from the end | does adding the facts help? |
| `"spread"` | `MAX_CHUNKS` chunks spread evenly over the **whole** case | does the middle (arguments) help? (needs tokenising the full text: slower) |

| `AGG` | document vector | trainable extras |
|---|---|---|
| `"mean"` | `d = mean_c h_c` | none |
| `"attn"` (default) | `α_c = softmax_c( vᵀ tanh(W h_c) )`, `d = Σ_c α_c h_c` | `W, v` (262k) — the learned `α` is also logged per chunk slot, an *evidence for which part of the judgment the model relies on* |
| `"hier"` | a small Transformer encoder over `[CLS, h_1 … h_K]` (+ learned slot embeddings) | ≈ 2.1M per layer |

### Why the Dec-02 attention can be unmasked (`ATTN_MODE="bidir"`)
Dec-02 uses RoPE, so position is carried by the rotation, not by the causal mask; removing the mask is legal (same argument as in Phase 3). Local layers become a **symmetric** band `|i−j| < 256`, global layers (3, 7, 11) see the whole chunk. `"causal"` keeps the Phase-1 mask (control).

### LoRMA (Bihany, Patel & Modi, ACL Findings 2025) in two lines
Instead of the additive update of LoRA, `W₀x` is *multiplied*: `h = I(BA)·(W₀ x)` with `A ∈ R^{r×d_out}`, `B ∈ R^{d_out×r}`. `variant="plus"` (default) is `I + (α/r)BA` with `B = 0` at init, so the adapted model equals the Phase-1 model at step 0 (checked in the notebook). `variant="pi"` rank-inflates `BA` by a cyclic row rotation (identity at init as well).

### What this notebook writes
`submission.zip` (verified like the evaluator would), `run_log.json` (parameters, peak GPU memory, train time, inference seconds per case, dev metrics, per-class P/R/F1, confusion matrix, per-slot attention profile, FP/FN by length bucket), `dev_errors.json`, and (if `EXPORT_SOFT_LABELS`) the teacher's document-level log-probabilities for the KD run.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p4_dec02_lorma"
METHOD     = "lorma"          # "lorma" | "lora" | "full" | "norm_only"
SMOKE_TEST = False            # True: ~150 steps on a small random subset, dev check only, NO submission written
ATTN_MODE  = "bidir"          # "bidir" (RoPE => unmasking is legal) | "causal" (Phase-1 mask)

# ---- Phase-1 checkpoint (written by caps_decoder_only_architecture_02.ipynb: runs/arch02_hybrid_gqa/best.pt) ----------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-02-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_4_CJPE/{Train/train.json, Train/dev.json, Test/test.json}
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_4_CJPE/ (checked first)
FINAL_FIT  = False                                # True: train on train+dev, NO validation (use after the dev run, EPOCHS = best epoch)

# ---- Long-document representation -----------------------------------------------------------
DOC_VIEW     = "tail"        # "tail" | "head" | "head_tail" | "spread"   (see the first markdown cell)
CHUNK_TOKENS = 512           # tokens per chunk INCLUDING the leading [BOS]  (<= 1024, the Phase-1 context)
MAX_CHUNKS   = 4             # chunks per case  => at most MAX_CHUNKS*(CHUNK_TOKENS-1) tokens are read
AGG          = "attn"        # "mean" | "attn" | "hier"
HIER_LAYERS  = 1             # AGG="hier": number of Transformer layers over the chunk vectors
CHAR_PER_TOK = 8             # tokenisation shortcut for long texts: only a slice of CHAR_PER_TOK chars per wanted token is tokenised
N_PROC       = 4             # worker processes for tokenisation (1 = serial)

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 4
BATCH_DOCS   = 4             # cases per micro-batch (each case = up to MAX_CHUNKS sequences)
ACCUM        = 4             # gradient accumulation  => effective batch = BATCH_DOCS*ACCUM cases
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
GRAD_CKPT    = False         # True: recompute block activations in the backward pass (less memory, ~30% slower)
TIME_BUDGET_MIN = None       # e.g. 150: stop training after this many minutes (the best epoch so far is kept)
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # variant: "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")   # LoRA and LoRMA
HEAD_DROPOUT    = 0.1
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
LABEL_SMOOTHING = 0.0

# ---- Decision threshold tuned on dev (macro-F1 post-processing) -----------------------------
THRESH_TUNE     = True
THRESH_MIN_GAIN = 0.005                           # keep the tuned threshold only if a 2-fold held-out check on dev gains this much macro-F1
THRESH_FINAL    = None                            # used when FINAL_FIT=True: paste the bias printed by the dev run (or None)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 800, 200

# ---- Knowledge distillation (run #5 of the Phase-4 plan): this run is the planned TEACHER ----------------------------------
EXPORT_SOFT_LABELS = True         # full run: save document-level log-probabilities of the train and dev cases

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE in ("causal", "bidir")
assert DOC_VIEW in ("tail", "head", "head_tail", "spread") and AGG in ("mean", "attn", "hier")
assert 16 <= CHUNK_TOKENS <= 1024 and MAX_CHUNKS >= 1, "chunks longer than 1024 exceed the Phase-1 RoPE table"
BODY = CHUNK_TOKENS - 1                           # tokens of text per chunk (the first position is [BOS])
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | ATTN_MODE={ATTN_MODE} | view={DOC_VIEW} {MAX_CHUNKS}x{CHUNK_TOKENS} | agg={AGG} | "
      f"batch={BATCH_DOCS}x{ACCUM} | epochs={EPOCHS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

RUN p4_dec02_lorma | METHOD=lorma | ATTN_MODE=bidir | view=tail 4x512 | agg=attn | batch=4x4 | epochs=4 | final_fit=False | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# Sentences are tokenised exactly as given (no clean_text / normalisation).
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_02.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model config (serialisable; saved inside every checkpoint)
@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                 # < n_head  => GQA (each KV head is shared by n_head//n_kv_head query heads)
    head_dim: int = 64
    ff_mult: float = 8 / 3             # SwiGLU hidden size ~ (8/3)*d_model, rounded up to a multiple of 64
    d_ff: Optional[int] = None         # if set, overrides ff_mult
    max_seq_len: int = 1024

    # --- attention pattern ---
    local_window: int = 0              # 0 => every layer is full causal attention
    global_layers: List[int] = field(default_factory=list)   # 0-indexed layers that stay FULL when local_window > 0

    # --- components ---
    norm_type: str = "rmsnorm"         # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"             # "rope" | "none"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"           # "swiglu" | "gelu"
    mixer_type: str = "attention"      # token-mixer registry key (swap in other mixers later)
    channel_mixer_type: str = "mlp"    # channel-mixer registry key

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)
        if self.local_window > 0:
            assert all(0 <= i < self.n_layer for i in self.global_layers)

    def is_global(self, layer_idx: int) -> bool:
        return self.local_window <= 0 or layer_idx in self.global_layers

    def window_for(self, layer_idx: int) -> Optional[int]:
        return None if self.is_global(layer_idx) else self.local_window


def preset_baseline_rope(vocab_size=None, max_seq_len=1024):
    """Model 01's shape (MHA 8/8, d_ff 1408, all layers full) but with RoPE really applied.
    Use this to train a clean controlled baseline for the ablation if you have GPU time."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len)


def preset_hybrid_gqa(vocab_size=None, max_seq_len=1024):
    """Architecture 02: GQA (2 KV heads) + sliding window 256 on 9 of 12 layers + wider FFN to keep params equal."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len,
                       n_kv_head=2, d_ff=1664, local_window=256, global_layers=[3, 7, 11])

print(json.dumps(asdict(preset_hybrid_gqa()), indent=1))

{
 "vocab_size": 32000,
 "n_layer": 12,
 "d_model": 512,
 "n_head": 8,
 "n_kv_head": 2,
 "head_dim": 64,
 "ff_mult": 2.6666666666666665,
 "d_ff": 1664,
 "max_seq_len": 1024,
 "local_window": 256,
 "global_layers": [
  3,
  7,
  11
 ],
 "norm_type": "rmsnorm",
 "norm_eps": 1e-05,
 "pos_type": "rope",
 "rope_theta": 10000.0,
 "mlp_type": "swiglu",
 "mixer_type": "attention",
 "channel_mixer_type": "mlp",
 "bias": false,
 "attn_dropout": 0.0,
 "resid_dropout": 0.0,
 "embd_dropout": 0.0,
 "tie_embeddings": true,
 "init_std": 0.02,
 "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers
class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: scale by 1/rms(x) only (no mean centering, no bias). Computed in fp32 for stability."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g


def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm": return LayerNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":   return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
class RoPE(nn.Module):
    """Rotates each (even, odd) pair of a q/k vector by an angle proportional to its position.
    Then q_i . k_j depends only on the RELATIVE offset i-j. Tables are precomputed; math runs in fp32."""
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)       # [max_seq_len, head_dim/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):                      # x: [B,H,T,D]; cos,sin: [*,1,T,D/2]
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids):
        """q: [B,Hq,T,D], k: [B,Hkv,T,D], position_ids: [B,T] (absolute positions of these T tokens)."""
        cos = self.cos[position_ids].unsqueeze(1).float()   # [B,1,T,D/2]
        sin = self.sin[position_ids].unsqueeze(1).float()
        dt = q.dtype
        return (self._rotate(q.float(), cos, sin).to(dt),
                self._rotate(k.float(), cos, sin).to(dt))

In [8]:
# Cell 7 — Attention: GQA + optional sliding window + KV cache
_MASK_CACHE = {}

def causal_window_mask(T: int, S: int, window: Optional[int], dev) -> torch.Tensor:
    """Bool mask [1,1,T,S], True = query may attend to key.
    The T queries are the LAST T positions of the S keys (so it also works with a KV cache).
    Query at absolute position p attends keys j with  j <= p  and  (window is None or p - j < window),
    i.e. itself + the previous (window-1) tokens."""
    key = (T, S, window, str(dev))
    m = _MASK_CACHE.get(key)
    if m is None:
        qpos = torch.arange(S - T, S, device=dev).view(T, 1)
        kpos = torch.arange(S, device=dev).view(1, S)
        allowed = kpos <= qpos
        if window is not None:
            allowed = allowed & ((qpos - kpos) < window)
        m = allowed.view(1, 1, T, S)
        if T > 1:                       # cache the big square-ish masks only
            _MASK_CACHE[key] = m
    return m


class Attention(nn.Module):
    """Causal multi-head self-attention.
    - GQA: n_kv_head K/V heads, each shared by n_head // n_kv_head query heads (KV cache shrinks by that factor).
    - Local layers: each token sees only the last `window` tokens. Global layers: window=None (full causal).
    - KV cache stores the (already RoPE-rotated) K and V."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.cfg = cfg
        self.layer_idx = layer_idx
        self.window = cfg.window_for(layer_idx)
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)            # [B,n,T,D]

    def forward(self, x, rope=None, position_ids=None, attention_mask=None, kv_cache=None):
        """x: [B,T,d]; position_ids: [B,T]; attention_mask: [B,S] (1 keep / 0 pad, S = cache + T);
        kv_cache: (k_prev, v_prev) each [B,n_kv_head,S_prev,D]. Returns (out, new_cache)."""
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)

        if rope is not None:                                              # RoPE on the NEW q/k only
            q, k = rope(q, k, position_ids)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        S = k.shape[2]

        if self.n_kv_head != self.n_head:                                 # GQA: share each K/V head
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        window_binds = self.window is not None and S > self.window
        need_mask = (attention_mask is not None) or window_binds or (T > 1 and S != T)

        allowed = None
        if need_mask:
            allowed = causal_window_mask(T, S, self.window, x.device)     # [1,1,T,S]
            if attention_mask is not None:
                allowed = allowed & attention_mask[:, None, None, :].bool()
                # a left-padding query would see nothing (-> NaN). Let every query see itself (harmless for real tokens).
                qpos = torch.arange(S - T, S, device=x.device).view(T, 1)
                kpos = torch.arange(S, device=x.device).view(1, S)
                allowed = allowed | (kpos == qpos).view(1, 1, T, S)

        p = self.attn_dropout.p if self.training else 0.0
        if self.cfg.use_sdpa:
            if need_mask:
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=p, is_causal=False)
            else:                                                         # plain causal (global layer, training)
                out = F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=(T > 1))
        else:                                                             # manual reference path (used in tests)
            if allowed is None:
                allowed = causal_window_mask(T, S, None, x.device)
            scores = (q @ k.transpose(-2, -1)) * self.scale
            scores = scores.masked_fill(~allowed, float("-inf"))
            probs = self.attn_dropout(F.softmax(scores.float(), dim=-1).to(q.dtype))
            out = probs @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Feed-forward blocks
class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

In [10]:
# Cell 9 — Registries (plug in new token mixers / channel mixers later) + transformer block
TOKEN_MIXERS, CHANNEL_MIXERS = {}, {}

def register_token_mixer(name):
    def deco(fn): TOKEN_MIXERS[name] = fn; return fn
    return deco

def register_channel_mixer(name):
    def deco(fn): CHANNEL_MIXERS[name] = fn; return fn
    return deco

@register_token_mixer("attention")
def _build_attention(cfg, layer_idx):
    return Attention(cfg, layer_idx)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    if cfg.mlp_type == "swiglu": return SwiGLU(cfg)
    if cfg.mlp_type == "gelu":   return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


class Block(nn.Module):
    """Pre-norm block:  x = x + mixer(norm(x));  x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.norm1, self.norm2 = build_norm(cfg), build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg, layer_idx)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)
    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone, LM head, weight init
class Backbone(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None, kv_caches=None):
        B, T = input_ids.shape
        if position_ids is None:                       # default: continue after whatever is in the cache
            past = kv_caches[0][0].shape[2] if kv_caches is not None else 0
            position_ids = (past + torch.arange(T, device=input_ids.device)).unsqueeze(0).expand(B, T)
        x = self.emb_drop(self.tok_emb(input_ids))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, rope=self.rope, position_ids=position_ids, attention_mask=attention_mask,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    """Backbone + LM head tied to the token embedding."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, position_ids=None, kv_caches=None):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      position_ids=position_ids, kv_caches=kv_caches)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss, new_caches


def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, init_std) everywhere; residual-branch output projections scaled by 1/sqrt(2*n_layer)."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            if hasattr(blk.mixer, "o_proj"):  blk.mixer.o_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "down_proj"): blk.cmix.down_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "fc2"):       blk.cmix.fc2.weight.mul_(scale)

In [12]:
# Cell 11 — Utilities: parameter count, optimizer groups (FIXED), save / load
def count_params(model: nn.Module) -> Dict[str, int]:
    """Tied embedding counted once (model.parameters() de-duplicates shared tensors)."""
    total = sum(p.numel() for p in model.parameters())
    emb = model.backbone.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Weight decay on 2-D weight matrices only (attention/MLP projections).
    No decay on norm gains/biases (1-D) and on the embedding.
    (Model 01 matched '.b' in names, which also matched 'backbone.blocks...' -> decay was never applied.)"""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model, cfg, extra=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, tmp)
    os.replace(tmp, path)


def load_model(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [13]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} | layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} | "
      f"heads {lm_cfg.n_head}:{lm_cfg.n_kv_head} (GQA) | d_ff={lm_cfg.d_ff} | local_window={lm_cfg.local_window} "
      f"global_layers={lm_cfg.global_layers} | params {pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE == "bidir" and lm_cfg.pos_type == "none":
    raise RuntimeError("ATTN_MODE='bidir' needs positional information that does not come from the causal mask, "
                       "but this checkpoint has pos_type='none'. Use ATTN_MODE='echo' or 'causal'.")

Phase-1 model | pos_type=rope | layers=12 d_model=512 | heads 8:2 (GQA) | d_ff=1664 | local_window=256 global_layers=[3, 7, 11] | params 54.93M (embedding 16.38M)
checkpoint extra: step 12749 | val_loss 1.9441224113106728
LM loss on a legal sentence: 1.888  (ln V = 10.37)


## Phase-4 data: cases, labels, statistics
Cases are `{"id", "text"}` (+ `"label"` for train/dev). The loader accepts a few equivalent layouts (a list, or a dict keyed by id; `text` may be a string or a list of strings; the label key may be `label`/`labels`/`y`/`judgment`/`outcome`), and **fails loudly** if a label is not 0/1. The statistics cell tells you how much of a typical case your `MAX_CHUNKS × CHUNK_TOKENS` budget covers. If the schema printed below is not what you expect, paste it to Claude.

In [14]:
# Cell 13 — Phase-4 data: train / dev / test cases
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P4 = "Phase_4_CJPE"

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_4_CJPE folder and set LOCAL_DATA_DIR.") from ex

def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default

def parse_cases(raw, split, has_labels):
    """-> list of {uid, id, text, y|None, nchar}.  uid = '<split>::<position in the file>' (stable, used internally and by the KD export)."""
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"text": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ("id", "case_id", "doc_id", "document_id"), i)
        text = _first(e, ("text", "case_text", "document", "judgment_text", "content"))
        if isinstance(text, list): text = "\n".join(str(t) for t in text)
        if not isinstance(text, str):
            raise ValueError(f"{split} case {rid}: cannot find the case text. Keys: {list(e)}")
        y = None
        if has_labels:
            lab = _first(e, ("label", "labels", "y", "judgment", "outcome", "decision"))
            try: y = int(lab)
            except Exception: raise ValueError(f"{split} case {rid}: label {lab!r} is not 0/1. Keys: {list(e)}")
            if y not in (0, 1): raise ValueError(f"{split} case {rid}: label {y} is not 0/1")
        docs.append(dict(uid=f"{split}::{i}", id=rid, text=text, y=y, nchar=len(text)))
    return docs

def load_split(rel, split, has_labels):
    raw = json.loads(fetch(rel).read_text(encoding="utf-8"))
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_cases(raw, split, has_labels)

TRAIN_ALL = load_split(f"{P4}/Train/train.json", "train", True)
DEV_ALL   = load_split(f"{P4}/Train/dev.json",   "dev",   True)
TEST_ALL  = load_split(f"{P4}/Test/test.json",   "test",  False)

for name, ds in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
    nc = np.asarray([d["nchar"] for d in ds])
    msg = f"{name:5s}: {len(ds):6d} cases | chars/case mean {nc.mean():9.0f} median {np.median(nc):9.0f} p90 {np.percentile(nc, 90):9.0f} max {nc.max():9d}"
    if ds[0]["y"] is not None: msg += f" | label counts {dict(sorted(Counter(d['y'] for d in ds).items()))}"
    print(msg)
for name, ds in (("dev", DEV_ALL), ("test", TEST_ALL)):
    ids = [str(d["id"]) for d in ds]
    if len(set(ids)) != len(ids): print(f"WARNING: {name} ids are not unique -> paste this to Claude (the submission needs one prediction per id)")
_tr_ids, _te_ids = {str(d["id"]) for d in TRAIN_ALL}, {str(d["id"]) for d in TEST_ALL}
print("train/test id overlap:", len(_tr_ids & _te_ids))

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/train.json: list of 37387 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/dev.json: list of 1988 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Test/test.json: list of 1517 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train:  37387 cases | chars/case mean     19531 median     13023 p90     38996 max   2492701 | label counts {0: 22067, 1: 15320}
dev  :   1988 cases | chars/case mean     22110 median     16700 p90     40495 max    260198 | label counts {0: 994, 1: 994}
test :   1517 cases | chars/case mean     22698 median     17320 p90     42859 max    233000
train/test id overlap: 0


In [15]:
# Cell 14 — Length statistics: how much of a case does the chunk budget cover? (tokenises a small random sample of full texts)
_rng = random.Random(SEED)
_sample = _rng.sample(TRAIN_ALL, min(150, len(TRAIN_ALL)))
_t0 = time.time()
_tok_lens = np.asarray([len(tokenizer.encode(d["text"])) for d in _sample])
_dt = time.time() - _t0
_chars = np.asarray([d["nchar"] for d in _sample])
print(f"sample of {len(_sample)} train cases | chars/token {(_chars.sum() / _tok_lens.sum()):.2f} | full-text tokenisation {_chars.sum() / _dt / 1e6:.2f} M chars/s "
      f"({len(_sample) / _dt:.1f} cases/s serial)")
print(f"tokens/case: median {np.median(_tok_lens):.0f} | p75 {np.percentile(_tok_lens, 75):.0f} | p90 {np.percentile(_tok_lens, 90):.0f} | "
      f"p99 {np.percentile(_tok_lens, 99):.0f} | max {_tok_lens.max()}")
_budget = MAX_CHUNKS * BODY
print(f"budget {MAX_CHUNKS} x {BODY} = {_budget} tokens: the whole case fits for {100 * (_tok_lens <= _budget).mean():.0f}% of the sample; "
      f"on average the view reads {100 * np.mean(np.minimum(_tok_lens, _budget) / _tok_lens):.0f}% of a case's tokens")
_est_full = len(TRAIN_ALL) / (len(_sample) / _dt)
print(f"serial full-text tokenisation of all train cases would take ~{_est_full / 60:.0f} min; "
      f"DOC_VIEW={DOC_VIEW!r} " + ("tokenises only slices of the text (much faster)." if DOC_VIEW != "spread" else "needs the full text (use N_PROC > 1)."))
if _budget < np.median(_tok_lens): print("note: the budget is below the median case length; that is the point of the view ablation (what part to read).")
_ = _sample, _tok_lens, _chars

sample of 150 train cases | chars/token 4.66 | full-text tokenisation 3.34 M chars/s (162.4 cases/s serial)
tokens/case: median 2752 | p75 4880 | p90 6928 | p99 41351 | max 88617
budget 4 x 511 = 2044 tokens: the whole case fits for 38% of the sample; on average the view reads 70% of a case's tokens
serial full-text tokenisation of all train cases would take ~4 min; DOC_VIEW='tail' tokenises only slices of the text (much faster).
note: the budget is below the median case length; that is the point of the view ablation (what part to read).


## Selecting and tokenising the chunks
Only what the view needs is tokenised: for `tail` / `head` / `head_tail` the notebook tokenises a **character slice** of `CHAR_PER_TOK` characters per wanted token (doubling it if the slice holds too few tokens), which is orders of magnitude cheaper than tokenising a whole judgment. Chunks are aligned to the **end** of the text for the tail (so the last chunk always ends exactly at the end of the case) and to the start for the head. Results are cached on disk and tokenisation runs in `N_PROC` worker processes.

In [16]:
# Cell 15 — Chunk selection (views), tokenisation, cache
def _enc(s):
    return list(tokenizer.encode(s))

def _chunks(ids, from_end):
    """Split a token list into pieces of BODY tokens; alignment from the end (tail) or from the start (head)."""
    n, out = len(ids), []
    if from_end:
        e = n
        while e > 0:
            s = max(0, e - BODY); out.append(ids[s:e]); e = s
        out.reverse()
    else:
        for s in range(0, n, BODY): out.append(ids[s:s + BODY])
    return out

def _slice_tokens(text, need, side):
    """First / last `need` tokens of `text`, tokenising only a character slice. Returns (tokens, text_was_fully_tokenised)."""
    m = need * CHAR_PER_TOK
    while True:
        if m >= len(text):
            ids = _enc(text); return (ids[:need] if side == "head" else ids[-need:]), True
        if side == "head":
            s = text[:m]; j = s.rfind(" "); s = s[:j] if j > m // 2 else s            # drop a cut-off last word
        else:
            s = text[-m:]; j = s.find(" "); s = s[j + 1:] if 0 <= j < m // 2 else s   # drop a cut-off first word
        ids = _enc(s)
        if len(ids) >= need + 8:
            return (ids[:need] if side == "head" else ids[-need:]), False
        m *= 2

def _view_from_full(ids):
    K = MAX_CHUNKS
    if DOC_VIEW == "tail":  return _chunks(ids, True)[-K:]
    if DOC_VIEW == "head":  return _chunks(ids, False)[:K]
    if DOC_VIEW == "head_tail":
        if len(ids) <= K * BODY: return _chunks(ids, False)
        Kh = (K + 1) // 2; Kt = K - Kh
        return _chunks(ids[:Kh * BODY], False) + _chunks(ids[-Kt * BODY:], True)
    P = _chunks(ids, False)                                                          # "spread": K chunks evenly over the whole case
    if len(P) <= K: return P
    return [P[i] for i in sorted(set(np.linspace(0, len(P) - 1, K).round().astype(int).tolist()))]

def select_chunks(text):
    K, n = MAX_CHUNKS, len(text)
    if DOC_VIEW == "spread" or n <= K * BODY * CHAR_PER_TOK:
        pieces = _view_from_full(_enc(text))
    elif DOC_VIEW == "tail":
        ids, _ = _slice_tokens(text, K * BODY, "tail"); pieces = _chunks(ids, True)[-K:]
    elif DOC_VIEW == "head":
        ids, _ = _slice_tokens(text, K * BODY, "head"); pieces = _chunks(ids, False)[:K]
    else:                                                                            # head_tail
        Kh = (K + 1) // 2; Kt = K - Kh
        h, c1 = _slice_tokens(text, Kh * BODY, "head"); t, c2 = _slice_tokens(text, Kt * BODY, "tail")
        pieces = _view_from_full(_enc(text)) if (c1 or c2) else _chunks(h, False)[:Kh] + _chunks(t, True)[-Kt:]
    if not pieces: pieces = [[EOS_ID]]                                               # an empty case still needs one token
    return pieces

_TXT = None
def _work(i):
    return [np.asarray([BOS_ID] + list(p), dtype=np.int32) for p in select_chunks(_TXT[i])]

def tokenize_docs(docs, cache_name):
    """Adds d['chunks'] = list of int32 arrays (each starts with [BOS], length <= CHUNK_TOKENS). Cached on disk."""
    global _TXT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"{cache_name}_v{VOCAB_SIZE}_{DOC_VIEW}_c{CHUNK_TOKENS}_k{MAX_CHUNKS}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None, None))[0] != d["nchar"]]
    if todo:
        _TXT = [d["text"] for d in docs]
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, todo, chunksize=8), total=len(todo), desc=f"tokenise {cache_name}", leave=False))
            except Exception as ex:
                print("parallel tokenisation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(todo, desc=f"tokenise {cache_name}", leave=False)]
        for i, r in zip(todo, res): cache[docs[i]["uid"]] = (docs[i]["nchar"], r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d["chunks"] = cache[d["uid"]][1]
    return docs

def chunk_stats(docs, title):
    nch = np.asarray([len(d["chunks"]) for d in docs]); ntok = np.asarray([sum(len(c) for c in d["chunks"]) for d in docs])
    print(f"{title}: {len(docs)} cases | chunks/case mean {nch.mean():.2f} (full {MAX_CHUNKS}: {100 * (nch == MAX_CHUNKS).mean():.0f}%) | "
          f"tokens/case read mean {ntok.mean():.0f}")

def slot_offset(n_chunks):
    """Slot of the first chunk. For the tail view the chunks are RIGHT-aligned, so the last chunk (end of the judgment) is always slot K-1."""
    return (MAX_CHUNKS - n_chunks) if DOC_VIEW == "tail" else 0

def build_batch(docs, idxs, with_labels=True):
    """-> ids [N, L], mask [N, L], owner [N] (case position in the batch), slot [N], y [B]."""
    seqs, owner, slot, ys = [], [], [], []
    for b, di in enumerate(idxs):
        d = docs[di]; off = slot_offset(len(d["chunks"]))
        for j, c in enumerate(d["chunks"]):
            seqs.append(c); owner.append(b); slot.append(off + j)
        if with_labels: ys.append(d["y"])
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), L), dtype=np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
    y = np.asarray(ys, dtype=np.int64) if with_labels else None
    return ids, mask, np.asarray(owner), np.asarray(slot), y

## Metrics, decision threshold (pure numpy)
Binary macro-F1 = mean of the F1 of class 0 and class 1. The classifier outputs `log p(1) − log p(0)` (the logit margin); predicting `1` when `margin + b > 0` with a tuned `b` is the binary version of the per-class bias tuning of Phase 3. `b` is kept only if a 2-fold, label-stratified check on dev (tune on one half, score the other) shows a real gain.

In [17]:
# Cell 16 — Metrics (macro-F1), confusion, threshold tuning
def bin_scores(y, p):
    y = np.asarray(y, dtype=np.int64); p = np.asarray(p, dtype=np.int64)
    cm = np.zeros((2, 2), dtype=np.int64); np.add.at(cm, (y, p), 1)                  # cm[gold, pred]
    tp = np.diag(cm).astype(np.float64); sup = cm.sum(1).astype(np.float64); npred = cm.sum(0).astype(np.float64)
    P = np.divide(tp, npred, out=np.zeros(2), where=npred > 0)
    R = np.divide(tp, sup,   out=np.zeros(2), where=sup > 0)
    Fs = np.divide(2 * P * R, P + R, out=np.zeros(2), where=(P + R) > 0)
    return dict(P=P, R=R, F=Fs, support=sup, cm=cm, accuracy=float(tp.sum() / max(cm.sum(), 1)),
                macro_f1=float(Fs.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()))

def margin(logp):
    return logp[:, 1] - logp[:, 0]

def evaluate(docs, logp, bias=0.0):
    y = np.asarray([d["y"] for d in docs])
    return bin_scores(y, (margin(logp) + bias > 0).astype(np.int64))

def tune_threshold(docs, logp, grid=np.arange(-3.0, 3.01, 0.1)):
    """Best additive bias b on the margin (ties -> the smallest |b|)."""
    y = np.asarray([d["y"] for d in docs]); m = margin(logp)
    base = bin_scores(y, (m > 0).astype(np.int64))["macro_f1"]; best_b, best = 0.0, base
    for b in sorted(grid.tolist(), key=abs):
        s = bin_scores(y, (m + b > 0).astype(np.int64))["macro_f1"]
        if s > best + 1e-9: best_b, best = float(b), s
    return best_b, base, best

def cv_threshold_gain(docs, logp, folds=2):
    """Honest check: tune on half of the dev cases, score the other half (label-stratified split); mean held-out gain."""
    y = np.asarray([d["y"] for d in docs]); fold = np.zeros(len(docs), dtype=int)
    for c in (0, 1):
        idx = np.where(y == c)[0]; fold[idx] = np.arange(len(idx)) % folds
    gains = []
    for f in range(folds):
        tr, te = np.where(fold != f)[0], np.where(fold == f)[0]
        b, _, _ = tune_threshold([docs[i] for i in tr], logp[tr])
        base = evaluate([docs[i] for i in te], logp[te])["macro_f1"]; tuned = evaluate([docs[i] for i in te], logp[te], b)["macro_f1"]
        gains.append(tuned - base)
    return float(np.mean(gains))

## Model, adaptation methods (LoRMA / LoRA / full / norm-only), training
The bidirectional-attention patch and the LoRA / LoRMA layers are the Phase-3 code, unchanged. New: the **document model** (chunk encoder → chunk pooling → aggregator → head), which replaces the sentence-pooling head.

In [18]:
# Cell 17 — Bidirectional local/global attention (as in Phase 3), the document model, and the adaptation methods
import copy

# ---- (a) Bidirectional attention for a LOCAL/GLOBAL decoder (identical to the Phase-3 notebook) ----------------------------
if "_causal_window_mask_orig" not in globals():
    _causal_window_mask_orig = causal_window_mask             # the Phase-1 function (kept so the mode can be switched back)
_BIDIR_CACHE = {}
def _bidir_window_mask(T, S, window, dev):
    """Bool [1,1,T,S], True = may attend.  Global layer (window=None): everything.  Local layer: |p - j| < window (symmetric band)."""
    key = (T, S, window, str(dev))
    m = _BIDIR_CACHE.get(key)
    if m is None:
        if window is None:
            m = torch.ones(1, 1, T, S, dtype=torch.bool, device=dev)
        else:
            qpos = torch.arange(S - T, S, device=dev).view(T, 1)
            kpos = torch.arange(S, device=dev).view(1, S)
            m = ((qpos - kpos).abs() < window).view(1, 1, T, S)
        if len(_BIDIR_CACHE) > 32: _BIDIR_CACHE.clear()
        _BIDIR_CACHE[key] = m
    return m
causal_window_mask = _bidir_window_mask if ATTN_MODE == "bidir" else _causal_window_mask_orig
print("attention masks:", "BIDIRECTIONAL (symmetric local band + full global)" if ATTN_MODE == "bidir" else "causal (Phase-1)")


# ---- (b) Document model: Phase-1 backbone per chunk -> mean-pool -> aggregator over chunks -> Linear(d, 2) ------------------
class DocModel(nn.Module):
    """Every chunk goes through the (adapted) Phase-1 backbone independently; its token vectors are mean-pooled (padding excluded).
    The chunk vectors of a case sit in slots 0..K-1 (right-aligned for the tail view) and are aggregated by AGG."""
    def __init__(self, backbone, d_model, n_slots, agg, dropout, hier_layers):
        super().__init__()
        self.backbone, self.agg, self.K = backbone, agg, n_slots
        if agg == "attn":
            self.att_w = nn.Linear(d_model, d_model); self.att_v = nn.Linear(d_model, 1, bias=False)
        elif agg == "hier":
            self.slot_emb = nn.Embedding(n_slots, d_model); nn.init.normal_(self.slot_emb.weight, 0.0, 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model)); self.ln_in = nn.LayerNorm(d_model); self.ln_out = nn.LayerNorm(d_model)
            layer = nn.TransformerEncoderLayer(d_model, 8, dim_feedforward=1024, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, hier_layers, enable_nested_tensor=False)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 2)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def encode(self, ids, mask):
        """Same computation as Backbone.forward (no KV cache), with optional activation checkpointing per block."""
        bb = self.backbone; N, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(N, T)
        x = bb.emb_drop(bb.tok_emb(ids))
        for blk in bb.blocks:
            if self.training and GRAD_CKPT:
                x = checkpoint(lambda t, blk=blk: blk(t, rope=bb.rope, position_ids=pos, attention_mask=mask)[0], x, use_reentrant=False)
            else:
                x, _ = blk(x, rope=bb.rope, position_ids=pos, attention_mask=mask)
        return bb.norm_f(x)

    def forward(self, ids, mask, owner, slot, B):
        h = self.encode(ids, mask).float()                                       # [N, T, d]
        m = mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp_min(1.0)                        # [N, d]   chunk vectors
        D = pooled.size(-1)
        grid = pooled.new_zeros(B, self.K, D); valid = torch.zeros(B, self.K, dtype=torch.bool, device=ids.device)
        grid[owner, slot] = pooled; valid[owner, slot] = True
        alpha = None
        if self.agg == "mean":
            v = valid.unsqueeze(-1).float(); doc = (grid * v).sum(1) / v.sum(1).clamp_min(1.0)
        elif self.agg == "attn":
            s = self.att_v(torch.tanh(self.att_w(grid))).squeeze(-1).masked_fill(~valid, float("-inf"))
            alpha = s.softmax(-1); doc = (alpha.unsqueeze(-1) * grid).sum(1)
        else:
            z = self.ln_in(grid) + self.slot_emb(torch.arange(self.K, device=ids.device)).unsqueeze(0)
            seq = torch.cat([self.cls.expand(B, 1, D), z], dim=1)
            kpm = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=ids.device), ~valid], dim=1)
            doc = self.ln_out(self.enc(seq, src_key_padding_mask=kpm)[:, 0])
        return self.head(self.drop(doc)), alpha


# ---- (c) LoRA (additive) --------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  W frozen; B = 0 at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


# ---- (d) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -----------------------------------------------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       variant 'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA ; computed as y + (alpha/r) B(A y), y = W0 x  -> O(b d r), same order as LoRA.
                                 init B = 0  => exactly the Phase-1 layer at step 0.
       variant 'pi'   (LoRMA_pi): I(BA) = rows of BA rotated cyclically (row i by i); init B[:,0]=1, A[0,0]=1, rest of A = 0
                                 => I(BA) = I_d at step 0 (the rotated 'ones x e0' matrix is the identity)."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)                                              # W0 x  (frozen), the multiplier acts on the OUTPUT space
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A                                           # [d, d], rank <= r
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))          # rank-inflated by the row rotation
        return y @ Tm.t()                                             # (Tm y) for every token


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


def build_model():
    """Fresh copy of the Phase-1 weights every call (several runs in one session do not contaminate each other)."""
    backbone = copy.deepcopy(lm.backbone).cpu()
    ref = copy.deepcopy(backbone) if METHOD in ("lora", "lorma") else None     # untouched copy for the identity check
    model = DocModel(backbone, lm_cfg.d_model, MAX_CHUNKS, AGG, HEAD_DROPOUT, HIER_LAYERS)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # everything trainable
    elif METHOD in ("lora", "lorma"):
        for p in backbone.parameters(): p.requires_grad = False
        if METHOD == "lora":
            n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
            print(f"LoRA applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        # The adapted model must equal the Phase-1 model at initialisation (B = 0 for LoRA / LoRMA+, rotated identity for LoRMA_pi).
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a, _ = backbone.eval()(_ids, attention_mask=_m); b, _ = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        backbone.train(); del ref
    elif METHOD == "norm_only":
        for p in backbone.parameters(): p.requires_grad = False
        n = 0
        for m in backbone.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_new = sum(p.numel() for n_, p in model.named_parameters() if not n_.startswith("backbone."))
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | of which new (aggregator + head) {n_new/1e6:.3f}M")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.bincount([d["y"] for d in train_docs], minlength=2).astype(np.float64)
    w = np.ones(2)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

attention masks: BIDIRECTIONAL (symmetric local band + full global)


In [19]:
# Cell 18 — Inference: cases -> log-probabilities (+ the attention profile over chunk slots)
@torch.no_grad()
def predict_logp(model, docs, batch_docs=None):
    """-> (logp [n, 2], alpha [n, K] or None, seconds). alpha[i, s] = attention weight of slot s (0 where the slot is empty)."""
    model.eval(); bs = batch_docs or 2 * BATCH_DOCS
    order = sorted(range(len(docs)), key=lambda i: sum(len(c) for c in docs[i]["chunks"]))     # similar lengths together
    out = np.zeros((len(docs), 2), dtype=np.float32); al = np.zeros((len(docs), MAX_CHUNKS), dtype=np.float32)
    t0 = time.time()
    for i in range(0, len(order), bs):
        idxs = order[i:i + bs]
        ids, mask, owner, slot, _ = build_batch(docs, idxs, with_labels=False)
        ids, mask = torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)
        owner, slot = torch.from_numpy(owner).to(device), torch.from_numpy(slot).to(device)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, alpha = model(ids, mask, owner, slot, len(idxs))
        out[idxs] = logits.float().log_softmax(-1).cpu().numpy()
        if alpha is not None: al[idxs] = alpha.float().cpu().numpy()
    if device.type == "cuda": torch.cuda.synchronize()
    return out, (al if model.agg == "attn" else None), time.time() - t0

In [20]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def train_doc_model(model, train_docs, val_docs, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, counts = class_weights(train_docs)
    print("class counts:", counts.astype(int).tolist(), "| class weights:", np.round(cw.cpu().numpy(), 3).tolist())
    new_p = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("backbone.") and p.requires_grad:
            (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=new_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(train_docs)
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH_DOCS) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} cases/epoch | {steps_per_epoch} optimiser steps/epoch (effective batch {BATCH_DOCS * ACCUM}) | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time(); n_seen = 0
    for ep in range(1, epochs + 1):
        order = list(range(n_items)); random.Random(SEED + ep).shuffle(order)
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(n_items / BATCH_DOCS)
        for bi in range(nb):
            idxs = order[bi * BATCH_DOCS:(bi + 1) * BATCH_DOCS]
            ids, mask, owner, slot, y = build_batch(train_docs, idxs)
            ids, mask, owner, slot, y = (torch.from_numpy(x).to(device) for x in (ids, mask, owner, slot, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits, _ = model(ids, mask, owner, slot, len(idxs))
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1; n_seen += len(idxs)
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step == 10:
                    per = (time.time() - t0) / 10
                    print(f"  >> speed: {per:.1f} s/step | {BATCH_DOCS * ACCUM / per:.1f} cases/s | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus dev evaluation)")
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25 * ACCUM:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping (the best epoch so far is kept)"); done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
            rec.update(val_macro_f1=ev["macro_f1"], val_acc=ev["accuracy"], val_P1=float(ev["P"][1]), val_R1=float(ev["R"][1]),
                       val_frac_pred1=float((lp[:, 1] > lp[:, 0]).mean()))
            if ev["macro_f1"] > best["score"]:
                best = dict(score=ev["macro_f1"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, cases_seen=n_seen,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [21]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TRAIN_ALL, min(SMOKE_TRAIN_DOCS, len(TRAIN_ALL)))
    val_docs = _r.sample(DEV_ALL, min(SMOKE_VAL_DOCS, len(DEV_ALL)))
elif FINAL_FIT:
    train_docs = TRAIN_ALL + DEV_ALL; val_docs = []
else:
    train_docs = TRAIN_ALL; val_docs = DEV_ALL
_t0 = time.time()
tokenize_docs(train_docs, "train_smoke" if SMOKE_TEST else ("train_dev" if FINAL_FIT else "train"))
if val_docs: tokenize_docs(val_docs, "dev_smoke" if SMOKE_TEST else "dev")
print(f"tokenisation {time.time() - _t0:.0f}s")
chunk_stats(train_docs, "train")
if val_docs: chunk_stats(val_docs, "dev")

if val_docs:                                        # reference points: constant predictions
    for c in (0, 1):
        _ev = evaluate(val_docs, np.log(np.tile(np.eye(2, dtype=np.float32)[c] * 0.998 + 0.001, (len(val_docs), 1))))
        print(f"constant-'{c}' baseline: dev macro-F1 {_ev['macro_f1']:.4f} | accuracy {_ev['accuracy']:.3f}")

model = build_model()
history, best, stats = train_doc_model(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev macro-F1 {best['score']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

tokenise train:   0%|          | 0/37387 [00:00<?, ?it/s]

tokenise dev:   0%|          | 0/1988 [00:00<?, ?it/s]

tokenisation 54s
train: 37387 cases | chunks/case mean 3.42 (full 4: 71%) | tokens/case read mean 1659
dev: 1988 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1949
constant-'0' baseline: dev macro-F1 0.3333 | accuracy 0.500
constant-'1' baseline: dev macro-F1 0.3333 | accuracy 0.500
LoRMA[plus] applied to 84 Linear layers (r=16, alpha=32)
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 57.16M | trainable 2.230M (3.90%) | of which new (aggregator + head) 0.264M
class counts: [22067, 15320] | class weights: [1.0, 1.2000000476837158]
37387 cases/epoch | 2337 optimiser steps/epoch (effective batch 16) | total 9348 steps
  >> speed: 1.2 s/step | 13.2 cases/s | projected 47.4 min/epoch (189 min for 4 epochs, plus dev evaluation)
  ep 1 step 25/9348 | loss 0.7181 | lr x0.045 | 29s
  ep 1 step 50/9348 | loss 0.7205 | lr x0.089 | 57s
  ep 1 step 75/9348 | loss 0.7234 | lr x0.134 | 86s
  ep 1 step 100/9348 | loss 0.7030 | lr x0.179 | 116s
  ep 1 step

In [22]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
    frac1 = float((lp[:, 1] > lp[:, 0]).mean())
    print(f"loss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | dev macro-F1 {ev['macro_f1']:.4f} | share predicted '1': {frac1:.2f}")
    ok = drop < 0.95 and 0.02 < frac1 < 0.98
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or one class dominates; ~150 steps is short for an adapter - compare METHOD='full' before debugging")
    print("Check the '>> speed' line above: it projects the minutes per epoch of the full run; set EPOCHS / MAX_CHUNKS / TIME_BUDGET_MIN accordingly.")
else:
    print("not a smoke run")

not a smoke run


In [23]:
# Cell 22 — Dev analysis: scores, per-class table, confusion, threshold tuning, FP/FN by length, attention profile (full runs with a dev set)
THRESH = 0.0 if THRESH_FINAL is None else float(THRESH_FINAL)
dev_ev = dev_ev_tuned = None; thresh_gain = 0.0; dev_extra = {}
if not SMOKE_TEST and val_docs:
    logp_val, alpha_val, t_inf = predict_logp(model, val_docs)
    dev_ev = evaluate(val_docs, logp_val)
    print(f"===== dev | macro-F1 {dev_ev['macro_f1']:.4f} | macro-P {dev_ev['macro_p']:.4f} | macro-R {dev_ev['macro_r']:.4f} | accuracy {dev_ev['accuracy']:.4f}   <- report the UNTUNED macro-F1")
    for c in (0, 1):
        print(f"  class {c}: support {int(dev_ev['support'][c]):5d} | P {dev_ev['P'][c]:.3f} | R {dev_ev['R'][c]:.3f} | F1 {dev_ev['F'][c]:.3f}")
    print("  confusion [gold x pred]:", dev_ev["cm"].tolist())
    print(f"  inference: {t_inf:.1f}s for {len(val_docs)} cases = {1000 * t_inf / len(val_docs):.0f} ms/case ({MAX_CHUNKS} chunks x {CHUNK_TOKENS} tokens max)")
    dev_extra["infer_ms_per_case"] = 1000 * t_inf / len(val_docs)

    if THRESH_TUNE:
        b_t, s0, s1 = tune_threshold(val_docs, logp_val)
        cv_gain = cv_threshold_gain(val_docs, logp_val) if len(val_docs) >= 20 else 0.0
        thresh_gain = cv_gain
        print(f"threshold tuning on dev: {s0:.4f} -> {s1:.4f} (in-sample gain {s1 - s0:+.4f}, optimistic) | 2-fold held-out gain {cv_gain:+.4f} | bias = {b_t:+.2f}")
        if cv_gain >= THRESH_MIN_GAIN:
            THRESH = b_t; dev_ev_tuned = evaluate(val_docs, logp_val, THRESH)
            print(f"  kept. tuned dev macro-F1 {dev_ev_tuned['macro_f1']:.4f}. For FINAL_FIT=True set  THRESH_FINAL = {THRESH:+.2f}")
        else:
            print(f"  held-out gain below THRESH_MIN_GAIN={THRESH_MIN_GAIN}: threshold NOT used")

    # --- characteristics of the errors: FP / FN by case length (character terciles) ---
    y_v = np.asarray([d["y"] for d in val_docs]); p_v = (margin(logp_val) + THRESH > 0).astype(int)
    nchar_v = np.asarray([d["nchar"] for d in val_docs]); cuts = np.percentile(nchar_v, [33.3, 66.7])
    bucket = np.digitize(nchar_v, cuts); names = ["short", "medium", "long"]
    rows = []
    print("\nerrors by case length (terciles of characters):")
    print(f"  {'bucket':8s} {'cases':>6s} {'median chars':>13s} {'acc':>6s} {'macroF1':>8s} {'FP':>5s} {'FN':>5s} {'frac gold=1':>12s}")
    for bk in range(3):
        sel = bucket == bk
        if sel.sum() == 0: continue
        s = bin_scores(y_v[sel], p_v[sel]); fp = int(((p_v == 1) & (y_v == 0) & sel).sum()); fn = int(((p_v == 0) & (y_v == 1) & sel).sum())
        rows.append(dict(bucket=names[bk], n=int(sel.sum()), median_chars=float(np.median(nchar_v[sel])), acc=s["accuracy"], macro_f1=s["macro_f1"], fp=fp, fn=fn, frac_gold1=float(y_v[sel].mean())))
        print(f"  {names[bk]:8s} {int(sel.sum()):6d} {np.median(nchar_v[sel]):13.0f} {s['accuracy']:6.3f} {s['macro_f1']:8.3f} {fp:5d} {fn:5d} {y_v[sel].mean():12.3f}")
    dev_extra["by_length"] = rows
    m_v = margin(logp_val) + THRESH
    dev_extra["mean_margin"] = dict(TP=float(m_v[(y_v == 1) & (p_v == 1)].mean()) if ((y_v == 1) & (p_v == 1)).any() else None,
                                    FP=float(m_v[(y_v == 0) & (p_v == 1)].mean()) if ((y_v == 0) & (p_v == 1)).any() else None,
                                    FN=float(m_v[(y_v == 1) & (p_v == 0)].mean()) if ((y_v == 1) & (p_v == 0)).any() else None,
                                    TN=float(m_v[(y_v == 0) & (p_v == 0)].mean()) if ((y_v == 0) & (p_v == 0)).any() else None)
    print("mean logit margin by outcome (a confident error has a large |margin|):", {k: (None if v is None else round(v, 2)) for k, v in dev_extra["mean_margin"].items()})
    errs = [dict(id=str(val_docs[i]["id"]), gold=int(y_v[i]), pred=int(p_v[i]), margin=float(m_v[i]), nchar=int(nchar_v[i]), n_chunks=len(val_docs[i]["chunks"]))
            for i in np.where(p_v != y_v)[0]]
    (RUN_DIR / "dev_errors.json").write_text(json.dumps(errs))

    # --- which chunk does the model rely on? mean attention weight per slot (AGG='attn') ---
    if alpha_val is not None:
        prof = alpha_val.mean(0); cnt = (alpha_val > 0).mean(0)
        where = {"tail": "slot K-1 = end of the judgment", "head": "slot 0 = start", "head_tail": "first slots = start, last slots = end", "spread": "slot 0 = start ... K-1 = end"}[DOC_VIEW]
        print(f"\nmean attention weight per chunk slot ({where}):", np.round(prof, 3).tolist(), "| share of cases with that slot filled:", np.round(cnt, 2).tolist())
        full = np.asarray([len(d["chunks"]) == MAX_CHUNKS for d in val_docs])
        if full.any(): print("   ... on cases with all slots filled:", np.round(alpha_val[full].mean(0), 3).tolist())
        dev_extra["attn_profile"] = prof.tolist(); dev_extra["attn_profile_full_cases"] = alpha_val[full].mean(0).tolist() if full.any() else None
else:
    print("no dev analysis (smoke run or FINAL_FIT)" + ("" if THRESH_FINAL is None else f" | using THRESH_FINAL={THRESH:+.2f}"))

===== dev | macro-F1 0.5419 | macro-P 0.6345 | macro-R 0.5845 | accuracy 0.5845   <- report the UNTUNED macro-F1
  class 0: support   994 | P 0.552 | R 0.889 | F1 0.682
  class 1: support   994 | P 0.716 | R 0.280 | F1 0.402
  confusion [gold x pred]: [[884, 110], [716, 278]]
  inference: 64.3s for 1988 cases = 32 ms/case (4 chunks x 512 tokens max)
threshold tuning on dev: 0.5419 -> 0.5781 (in-sample gain +0.0362, optimistic) | 2-fold held-out gain +0.0362 | bias = +0.40
  kept. tuned dev macro-F1 0.5781. For FINAL_FIT=True set  THRESH_FINAL = +0.40

errors by case length (terciles of characters):
  bucket    cases  median chars    acc  macroF1    FP    FN  frac gold=1
  short       662          8945  0.571    0.571   116   168        0.532
  medium      664         16700  0.566    0.556    98   190        0.494
  long        662         33629  0.613    0.601    88   168        0.474
mean logit margin by outcome (a confident error has a large |margin|): {'TP': 1.32, 'FP': 0.52, 'FN': 

In [24]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, logp, bias, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pred = (margin(logp) + bias > 0).astype(int)
    sub = [dict(id=d["id"], pred=int(p)) for d, p in zip(docs, pred)]            # the id keeps the type it has in test.json
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} cases | predicted share of '1': {pred.mean():.3f}")
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, one integer 0/1 per official test id, no extras."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, pred}"
    want = [d["id"] for d in docs]
    got = [e["id"] for e in sub]
    assert len(got) == len(set(map(str, got))), "duplicate ids in the submission"
    assert set(map(str, got)) == set(map(str, want)), f"ids differ from test.json: {len(set(map(str, want)) - set(map(str, got)))} missing, {len(set(map(str, got)) - set(map(str, want)))} unknown"
    assert all(isinstance(e["pred"], int) and not isinstance(e["pred"], bool) and e["pred"] in (0, 1) for e in sub), "pred must be the integer 0 or 1"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} cases | first entries {sub[:2]}")
    print("   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).")

if not SMOKE_TEST:
    tokenize_docs(TEST_ALL, "test")
    chunk_stats(TEST_ALL, "test")
    logp_test, _, _ = predict_logp(model, TEST_ALL)
    submission = write_submission(TEST_ALL, logp_test, THRESH, RUN_DIR / "submission")
    _tr = np.bincount([d["y"] for d in train_docs], minlength=2); print(f"train share of '1': {_tr[1] / _tr.sum():.3f}")
else:
    print("smoke run: no submission written")

tokenise test:   0%|          | 0/1517 [00:00<?, ?it/s]

test: 1517 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1947
wrote /kaggle/working/runs/p4_dec02_lorma/submission/submission.zip | 1517 cases | predicted share of '1': 0.396
VERIFIED /kaggle/working/runs/p4_dec02_lorma/submission/submission.zip | 1517 cases | first entries [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 1}]
   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).
train share of '1': 0.410


In [25]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / representation tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, Path): return str(o)
    return o

def _ev_summary(ev):
    if ev is None: return None
    return dict(macro_f1=ev["macro_f1"], macro_p=ev["macro_p"], macro_r=ev["macro_r"], accuracy=ev["accuracy"],
                per_class={str(c): dict(support=int(ev["support"][c]), P=round(float(ev["P"][c]), 4), R=round(float(ev["R"][c]), 4), F1=round(float(ev["F"][c]), 4)) for c in (0, 1)},
                confusion=ev["cm"].tolist())

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
           hier_layers=HIER_LAYERS, final_fit=FINAL_FIT, smoke=SMOKE_TEST, local_window=lm_cfg.local_window, pos_type=lm_cfg.pos_type,
           hp=HP[METHOD], epochs=EPOCHS, batch_docs=BATCH_DOCS, accum=ACCUM, class_weight=CLASS_WEIGHT,
           params_total=model.n_total, params_trainable=model.n_trainable, history=history, best_epoch=best["epoch"], stats=stats,
           threshold=THRESH, threshold_gain=thresh_gain, dev_untuned=_ev_summary(dev_ev), dev_tuned=_ev_summary(dev_ev_tuned), dev_extra=dev_extra,
           n_train=len(train_docs), n_dev=len(val_docs))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "agg": AGG, "doc_view": DOC_VIEW, "log": _jsonable(log)},
           RUN_DIR / f"cjpe_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['cjpe_p4_dec02_lorma.pt', 'dev_errors.json', 'run_log.json', 'submission']


In [26]:
# Cell 25 — Export this run's document-level distributions ("soft labels") so it can serve as the KD teacher (run #5)
# All four Phase-1 models share one tokenizer, so these arrays can teach ANY student without this notebook's model code.
# Stored as float16 log-probabilities: softmax(logp / T) = softmax(z / T) because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time(); _out, _split, _ids = {}, {}, {}
    for _name, _docs in (("train", train_docs), ("dev", val_docs)):
        if not _docs: continue
        _lp, _, _ = predict_logp(model, _docs)
        for d, l in zip(_docs, _lp):
            _out[d["uid"]] = l.astype(np.float16); _split[d["uid"]] = _name; _ids[d["uid"]] = str(d["id"])
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS), best_epoch=int(best["epoch"]),
                 final_fit=FINAL_FIT, dev_macro_f1=None if dev_ev is None else float(dev_ev["macro_f1"]), threshold=float(THRESH))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, docs=_out, split=_split, ids=_ids), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.1f} MB) in {time.time()-_t0:.0f}s | {len(_out)} cases | teacher params {_meta['params_total']/1e6:.1f}M | dev macro-F1 {_meta['dev_macro_f1']}")
    print("caveat: on the TRAIN cases the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

saved /kaggle/working/runs/p4_dec02_lorma/teacher_softlabels_p4_dec02_lorma.pt (5.3 MB) in 1127s | 39375 cases | teacher params 57.2M | dev macro-F1 0.5419441726259084
caveat: on the TRAIN cases the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.


## What to do next

1. **Smoke test first:** `SMOKE_TEST=True` (a few minutes). Read the *identity-at-init* line (LoRMA must equal Phase-1), the **length statistics** (Cell 14: how much of a case the budget covers), the **`>> speed` line** (projected minutes per epoch of the full run), the constant-prediction baselines and the smoke verdict.
2. **Choose the budget** from those numbers. If an epoch is too slow, lower `MAX_CHUNKS` (e.g. 3) or set `TIME_BUDGET_MIN`; if there is time to spare, raise `MAX_CHUNKS`. Keep `CHUNK_TOKENS=512` (the Phase-1 model saw 1024-token sequences; 512 keeps attention cheap).
3. **Full run:** `SMOKE_TEST=False`. It trains on train, selects the epoch on dev macro-F1, tunes the decision threshold (kept only if it helps on held-out dev halves), **already writes a valid `submission.zip`** and exports the teacher soft labels. Upload `submission.zip` to Codabench right away (evaluation takes 15–30 min).
4. **Controls for the report** (same notebook, change Cell 2 only, new `RUN_NAME` each time): `DOC_VIEW="head"` / `"head_tail"` / `"spread"` (which part of a judgment matters), `AGG="mean"` / `"hier"` (chunk-based vs hierarchical aggregation), `METHOD="full"` (can LoRMA match full fine-tuning?), `ATTN_MODE="causal"`. These are for the *analysis*; the 6 required submissions must be different **techniques**, not hyper-parameter variants.
5. **KD teacher:** with `EXPORT_SOFT_LABELS=True` this run already wrote `teacher_softlabels_<RUN_NAME>.pt`. Save it as a Kaggle dataset for the KD notebook.
6. Copy `run_log.json` (+ `dev_errors.json`) of every run into the report folder: parameters (total / trainable), peak GPU memory, train time, inference ms per case, per-class P/R/F1, confusion matrix, FP/FN by length, and the per-slot attention profile.